In [1]:
from pathlib import Path
import dask.array as da
import pandas as pd
import anndata as ad
import napari
import numpy as np
from tqdm.auto import tqdm
import json
import seaborn as sns
import matplotlib.pyplot as plt
expanded_piyg = ['#1a9641', '#a6d96a', '#978897', '#d1d1ca', '#f1b6da', '#d02c91']


def timed_compute(volume):
    """
    Compute a lazy Dask array frame-by-frame with progress reporting.

    This function iterates over the leading axis of a Dask array (e.g. time),
    calls `.compute()` on each slice, and stacks the results into a single
    NumPy array. A tqdm progress bar is displayed to indicate progress.

    Parameters
    ----------
    volume : dask.array.Array
        A Dask array with at least one dimension (e.g. shape (T, ...)).
        The function will iterate over the first axis (axis=0).

    Returns
    -------
    numpy.ndarray
        A NumPy array with the same shape as `volume`, but fully realized
        in memory. The dtype is preserved from the Dask array.

    Notes
    -----
    - Each frame is computed independently, which can be helpful for
      monitoring performance and memory use on large arrays.
    - The returned array may be very large if `volume` is large.
      Ensure sufficient memory is available.
    """
    return np.stack([frame.compute() for frame in tqdm(volume)], axis=0)


In [6]:
root_dir = Path('/mnt/OPERA3/Nathan/data/macrohet/Z_stack_tests/zarr')
# root_dir = Path('/Volumes/OPERA3/Nathan/Z_stack_tests/zarr')
rc_stem = "(3,3)"
store = root_dir / f"{rc_stem}.zarr"

# --- load images and segmentation ---
images = da.from_zarr(str(store / "images" / "0"))   # (T,C,Z,Y,X)
masks  = da.from_zarr(str(store / "labels" / "masks"))  # (T,Z,Y,X)
masks_tracked = da.from_zarr(str(store / "labels" / "masks_tracked"))  # (T,Z,Y,X)
# --- load tracks (CSV preferred, fallback to AnnData) ---
# tracks_csv = root_dir / f"{rc_stem}_tracks.csv"
# if tracks_csv.exists():
#     tracks = pd.read_csv(tracks_csv)
# else:
adata = ad.read_zarr(str(store / "tables" / "quantified_tracks"))
tracks = adata.obs.reset_index(drop=True)

print("images:", images.shape)
print("masks:", masks.shape)
print("tracked masks:", masks_tracked.shape)
print("tracks:", tracks.shape)


/home/dayn/miniconda3/envs/godspeed/lib/python3.10/site-packages/zarr/creation.py:250: UserWarning: ignoring keyword argument 'read_only'
  warn('ignoring keyword argument %r' % k)


images: (97, 2, 25, 7992, 7992)
masks: (97, 25, 7992, 7992)
tracked masks: (97, 7992, 7992)
tracks: (1859520, 10)


In [7]:
import dask.array as da
import numpy as np
from tqdm.auto import tqdm

# Assumes:
#   masks: dask array (T, Z, Y, X)
#   masks_tracked: dask array (T, Y, X)
#   z_ref: reference Z-plane index

z_ref = 12
T, Z, Y, X = masks.shape

# container for results, same chunks as masks
masks_tracked_Z = da.zeros_like(masks)

for t in tqdm(range(T), desc="Building tracked Z volumes"):
    # --- only compute the slices needed to build mapping ---
    old_ids = masks[t, z_ref].compute()        # (Y, X)
    new_ids = masks_tracked[t].compute()       # (Y, X)

    # build mapping from old -> new IDs
    pairs = np.stack([old_ids.ravel(), new_ids.ravel()], axis=1)
    pairs = pairs[pairs[:, 0] != 0]

    mapping = {}
    for old_id, new_id in pairs:
        mapping[int(old_id)] = int(new_id)

    # prepare LUT
    if mapping:
        max_id = int(np.max(list(mapping.keys())))
        lut = np.arange(max_id + 1, dtype=masks.dtype)
        for old_id, new_id in mapping.items():
            lut[old_id] = new_id
    else:
        lut = None  # just copy through

    # --- define function to apply mapping lazily ---
    def apply_mapping(block, lut=lut, mapping=mapping):
        if lut is not None and block.max() <= len(lut) - 1:
            return lut[block]
        elif mapping:
            out = block.copy()
            for old_id, new_id in mapping.items():
                out[block == old_id] = new_id
            return out
        else:
            return block

    # map over all z-slices for this timepoint
    result_t = masks[t].map_blocks(apply_mapping,
                                   dtype=masks.dtype,
                                   chunks=masks.chunksize[1:])
    masks_tracked_Z = da.concatenate(
        [masks_tracked_Z[:t], result_t[None], masks_tracked_Z[t+1:]]
    )

# at this point, masks_tracked_Z is a dask array, computed lazily
# you can persist to Zarr without loading everything in memory:
masks_tracked_Z.to_zarr("tracked_Z.zarr", overwrite=True)


Building tracked Z volumes:   0%|          | 0/97 [00:00<?, ?it/s]

ValueError: ('Shapes do not align: %s', [(1, 32, 8192, 8192), (96, 25, 7992, 7992)])

In [9]:
viewer

NameError: name 'viewer' is not defined